# TÜV SÜD source adapter

Control panel for the TÜV SÜD adapter (`tuev_sued` package: `1_scrapers/tuev_sued/src/`). All logic lives in the modules (`browser`, `parse`, `coverage`, `classify`, `run`, `report`, shared `core.radius_cover`); this notebook only calls them, and every path comes from `core.paths`. Build spec and recon notes: `0_instructions/sources/tuev_sued.md`.

**Python makes no request to TÜV SÜD.** tuvsud.com answers scripts with a Cloudflare challenge. Management approved (2026-10-05) running the queries in **your own, normal browser session** from the DevTools console: a generated snippet asks the finder (`POST …/searchTSCLocation`, one PLZ at a time, 5 s + 0–2 s apart, at most 200 per pass), downloads one `.jsonl` file per pass, and stops at once on any challenge. No headless browser, no automation tools, no faked headers, no challenge solver, no cookies copied into Python. **If TÜV SÜD challenges the in-browser requests too: stop and report. There is no workaround.**

**How completeness is proved.** The finder returns the **50 nearest** Service-Center to a search centre, sorted by distance, with no count field. Each answer therefore proves a disk around the server's centre (radius = distance of the last record − 0.5 km). Every regular PLZ centroid (8,309) must lie at least **15 km inside** some disk; the premise (sorted answers, no listed site missing from a disk that contains it) is checked as well.

**Scope.** `in_scope` = `huau == 1` and `tscStatus == 1` and country Germany → `pruefstelle`. Anything else → `unknown` + `needs_review`; flags (`uo_notice`, `coordinate_far_from_plz`) never drop a row. Partner-operated sites and TÜV Hessen / TÜV Hanse stations in the finder are Prüfstellen too (decision 2026-10-06): no flag, only the `operator_group` column.

**One monthly run, step by step** (about 10–15 min of manual work per pass):

1. Section 4: `plan` writes `tuev_sued_pass01.js` into a new run folder.
2. Chrome: open the HU finder, accept cookies, search one PLZ by hand. DevTools → Console → paste the file → Enter. It downloads `tuev_sued_pass01_<date>_<time>.jsonl`.
3. Section 5: `check` with that file. Result **PASS**, or **INCOMPLETE** with the next snippet (repeat 2), or FAIL / BLOCKED.
4. Section 6: `build` writes the exports and the run summary.

## 1. Setup and parameters

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from core import paths
from core.geo import ensure_geonames, load_plz_centroids
from tuev_sued import browser, coverage, report
from tuev_sued import run as tuev_run

ROOT = paths.project_root()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# ---- parameters (plain language) -------------------------------------------------------------------------------
EXPORTS = ("pruefstellen_csv",)   # only the in-scope CSV: no `all` file and no Excel (decision 2026-10-06); add "all_csv" or the xlsx keys only if you need them
cfg = coverage.CFG        # tie margin 0.5 km, cover margin 15 km, first-pass lattice 60 km, fill-in at most 60 PLZ per pass
print(f"cover margin {cfg.cover_margin_km:g} km | lattice spacing {cfg.lattice_spacing_km:g} km | fill-in cap {cfg.max_fill_per_pass} PLZ | max {coverage.MAX_PASSES} passes | max {browser.MAX_PLZ} PLZ per snippet")

centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))
print(f"{len(centroids)} regular PLZ centroids (GeoNames, CC BY 4.0)")

## 2. Before a run: passes, queries, time (simulated)

The simulated world (350 fake Service-Center, dense in Bayern and Baden-Württemberg) shows how the planning behaves; the real number of fill-in passes depends on where TÜV SÜD's sites really are. A later month starts with the PLZ list of the last passing run, which is normally enough for **one** pass.

In [ ]:
est = tuev_run.estimate(centroids)
print(f"pass 1 (lattice):        {est['planned_queries']} PLZ")
print(f"passes in the simulation: {est['expected_passes']}  {est['queries_per_pass']} PLZ per pass")
print(f"total queries:           {est['expected_queries']}")
print(f"browser time per pass:   ~{est['expected_minutes_per_pass']} min")
report.planned_map(centroids, est["planned_points"]);

## 3. Simulated run (offline)

The whole flow against a fake finder and a fake browser, through the same import and check code as a real pass. Must **PASS** and find 100 % of the fake sites.

In [ ]:
sim = tuev_run.run(mode="simulated", centroids=centroids, exports=EXPORTS)
RESULT = sim   # the run the results sections show; set RESULT = built after a real run, or RESULT = report.load_run("<run folder>")

### 3b. Rebuild (offline)

Rebuilds the exports and re-checks the coverage from the `raw.jsonl` of an earlier run. The new folder goes to `output/` next to its source; no copy of the raw file. Set `REBUILD_FROM` to a run folder name or path.

In [ ]:
REBUILD_FROM = None   # e.g. "2026-11-03_0900_full"
if REBUILD_FROM:
    rebuilt = tuev_run.run(mode="rebuild", centroids=centroids, source_run=REBUILD_FROM, exports=EXPORTS)
else:
    print("Rebuild skipped: set REBUILD_FROM to a run folder.")

## 4. Plan a pass

`plan` opens the run folder (`full` → `output/`, `sample` → `test_output/`) and writes the snippet of pass 1. **It makes no request.** Run it only when you are ready to paste the snippet in the browser.

* **sample**: 3 PLZ (80686 München, 24937 Flensburg, 01067 Dresden). First live test of the snippet; answers the open question *does a sparse area such as Flensburg also return exactly 50?* Never a PASS.
* **full**: the PLZ list of the last passing full run, otherwise the hex lattice (about 130 PLZ).

In [ ]:
PLAN_MODE = None   # "sample" or "full"; None = do nothing
if PLAN_MODE:
    PLAN = tuev_run.plan(PLAN_MODE, centroids=centroids)
    RUN_FOLDER = PLAN.run_folder
    print(f"run folder: {RUN_FOLDER}")
    print(f"snippet:    {PLAN.js_path}  ({len(PLAN.queries)} PLZ)")
else:
    print("Nothing planned: set PLAN_MODE = 'sample' or 'full'.")

## 5. Check a downloaded pass

Set `DOWNLOAD` to the file the snippet downloaded. `check` validates it (a bad file changes nothing), copies it into the run folder as `raw_passNN.jsonl`, and proves coverage over all passes so far. On **INCOMPLETE** it has written the next snippet; paste that one the same way and run this cell again with the new file.

In [ ]:
DOWNLOAD = None   # e.g. r"C:\Users\<you>\Downloads\tuev_sued_pass01_2026-11-03_0900.jsonl"
if DOWNLOAD:
    CHECK = tuev_run.check(RUN_FOLDER, DOWNLOAD, centroids=centroids)
    print(CHECK.message)
    if CHECK.next_plan:
        print("next snippet:", CHECK.next_plan.js_path)
else:
    print("Nothing to check: set DOWNLOAD to the downloaded .jsonl file.")

## 6. Build the exports

Merges the pass files into `raw.jsonl`, classifies, the export `tuev_sued_pruefstellen_<date>.csv` (the in-scope rows; no `all` file and no Excel), writes `run_summary.json/.md`, `query_log.csv` and, if targets stayed open, `open_targets.csv`. A run that is not a PASS labels its exports `grid-incomplete` or `unverified`. The drop check compares the in-scope count with the newest earlier **passing** full run (alert below −10 %). A folder is built once.

In [ ]:
BUILD = False   # True = build the run in RUN_FOLDER
if BUILD:
    built = tuev_run.build(RUN_FOLDER, centroids=centroids, exports=EXPORTS)
    RESULT = built
else:
    print("Not built: set BUILD = True after the last check.")

## 7. Results

Shows `RESULT` (the simulated run by default). Counts and maps contain no personal data for simulated runs; for real runs the review tables show names and the site emails: **do not save the notebook with those outputs**.

### 7a. Outcome and checks

In [ ]:
s = RESULT.summary
print(s["outcome"], "|", RESULT.run_folder.name)
for reason in s["status_reasons"]:
    print("  !", reason)
display(report.pass_table(s))
report.rule_table(s)

### 7b. Counts, per Bundesland, drop check

In [ ]:
display(report.counts_table(RESULT).to_frame("count"))
print("Drop check:", s["drop_check"])
if s.get("baseline"):
    print("Baseline:  ", s["baseline"])
print("Flags:", s["flags"])
print("Proven radius per answer (km):", s["coverage"].get("radius_km"), "| max site-to-PLZ-centroid distance:", s["coverage"].get("max_site_to_plz_centroid_km"), "km")
report.per_state(RESULT.frame)

### 7c. Published figure

What TÜV SÜD says about its network. The only figure found is from 2015; a current one still has to be found.

In [ ]:
report.published_comparison(s)

### 7d. Warnings (not failures)

Answers with fewer than 50 sites (a radius limit would change the picture), empty answers, sites far from every PLZ centroid (the proof assumes ≤ 10 km), address conflicts for one `tscId`, unreadable opening hours, notices without a time window, the unknown profile-page base address.

In [ ]:
for name, value in (s.get("warnings") or {}).items():
    if isinstance(value, dict) and "total" in value:
        print(f"{name}: {value['total']}")
        for item in value["items"][:5]:
            print("   ", item)
    else:
        print(f"{name}: {value}")
if not s.get("warnings"):
    print("no warnings")

### 7e. Fields seen in the answers

Every key of `location` with its fill rate. The schema is not final; this is what the first real run is read against (field names the parser had to guess: opening hours, notice windows, booking link, service flags).

In [ ]:
report.key_fill_table(s)

## 8. Maps

In [ ]:
report.coverage_map(RESULT, centroids);   # the proven disks per pass; black x = targets not covered

In [ ]:
report.sites_map(RESULT.frame);   # red = in scope, grey = outside the scope rule

## 9. Review tables (names and emails for real runs)

Everything stays in `tuev_sued_all`; these are the rows a human should look at: the Wolnzach-type notice (`uo_notice`) and every `needs_review` reason. `by_operator` shows who runs the sites (own, partner, TÜV Hessen, TÜV Hanse).

In [ ]:
tables = report.review_tables(RESULT.frame)
tables["needs_review_by_reason"]

In [ ]:
tables["uo_notice"]

In [ ]:
tables["by_operator"]